# Fine-tune Qwen with LoRA
`TEST_RUN = True` does a short test (a few steps, a checkpoint, a resume, the header check). For the real run
set it to `False`, or start it from a terminal with `TEST_RUN=0` in front of the command (see README step 5).

Memory on the laptop's RTX 3060 (6 GB): batch 1 needs ~4.3 GB for the longest pieces, batch 2 needs ~7.2 GB and
spills into normal RAM (very slow), and gradient checkpointing doesn't help because most memory goes to the scores
over Qwen's 151k-token vocabulary. So: batch 1, no checkpointing, gradient accumulation for 32 pieces per step.
On the laptop a full pass is ~10 hours or more, so it saves a checkpoint every ~50 min and picks up from the last one if you
stop it and run it again.

In [1]:
import os
import time
import torch
import datasets
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, PeftModel

MODEL_DIR = "../model/Qwen3-0.6B-Base"
DATA = "../data/train_test/"
OUT = "../model/"

TEST_RUN = os.environ.get("TEST_RUN", "1") != "0"   # "0" = real run
BATCH = int(os.environ.get("BATCH", "1"))   # 1 on the 6 GB laptop (2 doesn't fit), bigger GPUs can take more
PIECES_PER_STEP = 8 if TEST_RUN else 32   # batch size x gradient accumulation
assert PIECES_PER_STEP % BATCH == 0, f"BATCH has to divide {PIECES_PER_STEP}"
LR = 2e-4
SAVE_EVERY = 5 if TEST_RUN else 250       # steps; 250 steps is ~50 min
RUN_DIR = OUT + ('lora_test_runs/' if TEST_RUN else 'lora_runs/')   # checkpoints go here
FINAL = OUT + ('lora_test' if TEST_RUN else 'lora')
LOG = OUT + ('train_log_test.txt' if TEST_RUN else 'train_log.txt')  # progress, so you can watch it

# the model takes ~1.3 GB and each piece in a batch up to ~3 GB (the longest ones), so check there's room
free, total = torch.cuda.mem_get_info()
print(f"{'TEST RUN' if TEST_RUN else 'REAL RUN'} | GPU: {torch.cuda.get_device_name(0)}, {free / 1e9:.1f} of {total / 1e9:.1f} GB free")
need = 1.5 + 3 * BATCH
assert free / 1e9 > need, f"need ~{need:.0f} GB free GPU memory for BATCH={BATCH} - close other programs, use the other GPU or a smaller BATCH"

def log(msg):
    print(msg)
    with open(LOG, 'a') as f:
        f.write(time.strftime('%H:%M:%S') + ' ' + msg + '\n')

W1116 13:11:51.131000 13188 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


TEST RUN | GPU: NVIDIA GeForce RTX 3060 Laptop GPU, 5.4 of 6.4 GB free


## Load the data

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

cols = ['input_ids', 'attention_mask', 'is_end']
train = datasets.Dataset.from_parquet(DATA + 'train.parquet').select_columns(cols)
val = datasets.Dataset.from_parquet(DATA + 'val.parquet').select_columns(cols)
train_len = pc.list_value_length(pq.read_table(DATA + 'train.parquet', columns=['input_ids'])['input_ids']).to_numpy()

# random val sample for the evals (the file has all the poems first, so the first rows have no gutenberg)
val_sample = val.shuffle(seed=42).select(range(100 if TEST_RUN else 300))

print(len(train), "train pieces,", train_len.sum(), "train tokens |", len(val_sample), "val pieces for evals")

89344 train pieces, 27578174 train tokens | 100 val pieces for evals


## Batches
Qwen uses `<|endoftext|>` both for padding and for the end of a poem. The usual collator hides every
`<|endoftext|>` from the loss, so the model would never learn to end a poem. So the labels come from the
attention mask instead: padding is ignored (-100), everything else, including `<|endoftext|>`, is learned.

In [3]:
def collate(batch):
    longest = max(len(x['input_ids']) for x in batch)
    ids, mask, labels = [], [], []
    for x in batch:
        pad = longest - len(x['input_ids'])
        ids.append(x['input_ids'] + [tokenizer.pad_token_id] * pad)
        mask.append([1] * len(x['input_ids']) + [0] * pad)
        labels.append(x['input_ids'] + [-100] * pad)
    return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(mask), 'labels': torch.tensor(labels)}

# check on a real batch: an end piece next to a longer piece (so it gets padded)
end_i = next(i for i in range(len(train)) if train[i]['is_end'] and train_len[i] < 200)
b = collate([train[end_i], train[int(train_len.argmax())]])
n = int(train_len[end_i])
assert b['labels'][0, n - 1] == tokenizer.eos_token_id and (b['labels'][0, n:] == -100).all()
print("end-of-text is kept in the labels, padding is ignored")

end-of-text is kept in the labels, padding is ignored


## Model with LoRA

In [4]:
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to('cuda')
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 454.89it/s]


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


## Does the `Author:` header do anything?
For val pieces with a named poet, compare the loss on the poem itself (not the header) when the header has the
real name vs `Author: Unknown`. If the model has learned the poets' styles, the real name should give a lower
loss, so `gap = loss(Unknown) - loss(real name)` should grow above zero during training. This runs before training
and at every checkpoint.

In [5]:
val_text = pd.read_parquet(DATA + 'val.parquet', columns=['text', 'header'])
named = val_text[val_text['header'] != 'Unknown'].sample(50 if TEST_RUN else 200, random_state=42)

def poem_loss(model, text, name):
    ids = tokenizer(text)['input_ids']
    head = tokenizer(f"Author: {name}\n\n")['input_ids']
    if ids[:len(head)] != head:
        return None
    x = torch.tensor([ids]).to('cuda')
    labels = x.clone()
    labels[0, :len(head)] = -100   # only score the poem, not the header
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        return model(input_ids=x, labels=labels).loss.item()

def header_gap(model):
    was_training = model.training
    model.eval()
    real, unknown = [], []
    for text, name in zip(named['text'], named['header']):
        body = text.split('\n\n', 1)[1]
        r = poem_loss(model, text, name)
        u = poem_loss(model, 'Author: Unknown\n\n' + body, 'Unknown')
        if r is not None and u is not None:
            real.append(r)
            unknown.append(u)
    if was_training:
        model.train()
    r, u = sum(real) / len(real), sum(unknown) / len(unknown)
    return r, u, u - r

r, u, gap = header_gap(model)
log(f"header check before training: real name {r:.3f}, Unknown {u:.3f}, gap {gap:+.3f}")

header check before training: real name 4.198, Unknown 4.225, gap +0.026


## Train
Saves a checkpoint every `SAVE_EVERY` steps (keeps the last 3), checks the val loss and the header gap there, and
writes everything to the log file. If a checkpoint exists already, it carries on from there.

In [6]:
class Progress(TrainerCallback):
    def __init__(self):
        self.times = []
    def on_step_end(self, args, state, control, **kwargs):
        self.times.append(time.time())
    def on_log(self, args, state, control, logs=None, **kwargs):
        log(f"step {state.global_step}/{state.max_steps} {logs}")
    def on_save(self, args, state, control, model=None, **kwargs):
        r, u, gap = header_gap(model)
        log(f"checkpoint at step {state.global_step}: header check real {r:.3f}, Unknown {u:.3f}, gap {gap:+.3f}")

def make_trainer(max_steps):
    args = TrainingArguments(
        output_dir=RUN_DIR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=PIECES_PER_STEP // BATCH,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_steps=2 if TEST_RUN else 100,
        num_train_epochs=1,
        max_steps=max_steps,   # -1 = one full pass (the LR schedule is set over the whole pass)
        bf16=True,
        logging_steps=2 if TEST_RUN else 10,
        eval_strategy='steps',
        eval_steps=SAVE_EVERY,
        save_strategy='steps',
        save_steps=SAVE_EVERY,
        save_total_limit=3,    # model/ syncs to OneDrive, so don't pile up checkpoints
        report_to='none',
        dataloader_num_workers=0,
    )
    return Trainer(model=model, args=args, train_dataset=train, eval_dataset=val_sample,
                   data_collator=collate, callbacks=[progress])

progress = Progress()
os.makedirs(RUN_DIR, exist_ok=True)
last = get_last_checkpoint(RUN_DIR)
trainer = make_trainer(10 if TEST_RUN else -1)
if last:
    log(f"carrying on from {last}")
else:
    log(f"starting fresh, val loss {trainer.evaluate()['eval_loss']:.3f}")
trainer.train(resume_from_checkpoint=last)

step 0/0 {'eval_loss': 4.273749828338623, 'eval_model_preparation_time': 0.0228, 'eval_runtime': 23.1529, 'eval_samples_per_second': 4.319, 'eval_steps_per_second': 4.319, 'epoch': 0}


Training Loss,Validation Loss,Step
No log,4.273750,0


starting fresh, val loss 4.274


Step,Training Loss,Validation Loss
5,4.004517,4.233956
10,4.084952,4.217489


step 2/10 {'loss': 4.023678779602051, 'grad_norm': 1.6209049224853516, 'learning_rate': 0.0001, 'epoch': 0.00017908309455587392}
step 4/10 {'loss': 4.004517078399658, 'grad_norm': 0.9231594800949097, 'learning_rate': 0.0001923879532511287, 'epoch': 0.00035816618911174784}
step 5/10 {'eval_loss': 4.2339558601379395, 'eval_model_preparation_time': 0.0228, 'eval_runtime': 23.2908, 'eval_samples_per_second': 4.294, 'eval_steps_per_second': 4.294, 'epoch': 0.00044770773638968484}
checkpoint at step 5: header check real 4.171, Unknown 4.187, gap +0.015
step 6/10 {'loss': 4.016102313995361, 'grad_norm': 0.7901901006698608, 'learning_rate': 0.000138268343236509, 'epoch': 0.0005372492836676218}
step 8/10 {'loss': 4.107332229614258, 'grad_norm': 0.920960545539856, 'learning_rate': 6.173165676349103e-05, 'epoch': 0.0007163323782234957}
step 10/10 {'loss': 4.084952354431152, 'grad_norm': 0.8108494877815247, 'learning_rate': 7.612046748871327e-06, 'epoch': 0.0008954154727793697}
step 10/10 {'eval_l

TrainOutput(global_step=10, training_loss=4.047316551208496, metrics={'train_runtime': 163.6828, 'train_samples_per_second': 0.489, 'train_steps_per_second': 0.061, 'total_flos': 65672724480000.0, 'train_loss': 4.047316551208496, 'epoch': 0.0008954154727793697})

## How did it go?

In [7]:
history = trainer.state.log_history
evals = [(h['step'], round(h['eval_loss'], 3)) for h in history if 'eval_loss' in h]
print("val loss by step:", evals)
print("LoRA weights changed:", any(p.abs().sum() > 0 for n, p in model.named_parameters() if 'lora_B' in n))
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

gaps = sorted(b - a for a, b in zip(progress.times, progress.times[1:]))
if len(gaps) > 2:
    sec_per_step = gaps[len(gaps) // 2]   # typical step, so the pauses for evals/checkpoints don't count
    tokens_per_sec = PIECES_PER_STEP * train_len.mean() / sec_per_step
    print(f"typical step: {sec_per_step:.1f} s for {PIECES_PER_STEP} pieces")
    print(f"~{tokens_per_sec:.0f} tokens/s -> a full pass over {train_len.sum():,} tokens is ~{train_len.sum() / tokens_per_sec / 3600:.1f} h (plus evals)")

val loss by step: [(5, 4.234), (10, 4.217)]
LoRA weights changed: True
peak GPU memory: 3.89 GB
typical step: 6.4 s for 8 pieces
~388 tokens/s -> a full pass over 27,578,174 tokens is ~19.7 h (plus evals)


Test only: check that stopping and carrying on works (picks up the last checkpoint and does 2 more steps).

In [8]:
if TEST_RUN:
    last = get_last_checkpoint(RUN_DIR)
    trainer = make_trainer(12)
    trainer.train(resume_from_checkpoint=last)
    print("resumed from", os.path.basename(last), "-> now at step", trainer.state.global_step)
    assert trainer.state.global_step == 12

Step,Training Loss,Validation Loss
12,3.994827,4.217806


step 12/12 {'loss': 3.994826555252075, 'grad_norm': 0.8223726749420166, 'learning_rate': 4.8943483704846475e-06, 'epoch': 0.0010744985673352436}
step 12/12 {'eval_loss': 4.217805862426758, 'eval_runtime': 26.6295, 'eval_samples_per_second': 3.755, 'eval_steps_per_second': 3.755, 'epoch': 0.0010744985673352436}
checkpoint at step 12: header check real 4.159, Unknown 4.170, gap +0.011
step 12/12 {'train_runtime': 65.6559, 'train_samples_per_second': 0.244, 'train_steps_per_second': 0.03, 'total_flos': 77867581440000.0, 'train_loss': 3.994826555252075, 'epoch': 0.0010744985673352436}
resumed from checkpoint-10 -> now at step 12


## Save, load back, and write a few lines

In [9]:
model.save_pretrained(FINAL)
del trainer, model
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to('cuda')
loaded = PeftModel.from_pretrained(base, FINAL).eval()

torch.manual_seed(0)
for poet in ['Emily Dickinson', 'John Milton', 'Walt Whitman']:
    inputs = tokenizer(f"Author: {poet}\n\n", return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = loaded.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.95,
                              pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print('-----')

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 325.60it/s]


Author: Emily Dickinson
1.
The house I love is a little house,
Where all my things are safe.
But I can never know
Who has built it for me.
I know that the house is built,
By one who is not me,
Who has my heart
-----
Author: John Milton
John Milton, born in 1608 and baptized at the age of eighteen, was a renowned English poet. In the words of his father, William Milton, "I am a man of great and many talents, and a noble man,
-----
Author: Walt Whitman
AFTERWARD I saw the moon in the clouds. The sky was so full of light that the sky and the moon were one. The sun and the moon were one. The light from the sun and the moon were one. The moon was the
-----
